In [ ]:
import collections
import csv
import pathlib

import fancyimpute
import numpy as np
import sklearn.utils.extmath

In [ ]:
path_throughputs = pathlib.PurePath('outputs', 'links', '20260401000000.csv')
ie_pair_to_traffic = collections.defaultdict(float)
with open(path_throughputs, 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        ie_pair_to_traffic[row['source_id'], row['target_id']] += float(row['throughput'])

index_to_node = list(set(source for source, _ in ie_pair_to_traffic.keys()) | set(destination for _, destination in ie_pair_to_traffic.keys()))
node_to_index = {node: index for index, node in enumerate(index_to_node)}
n_nodes = len(index_to_node)

traffic_matrix = np.full((n_nodes, n_nodes), np.nan)
for (source, destination), traffic in ie_pair_to_traffic.items():
    traffic_matrix[node_to_index[source], node_to_index[destination]] = traffic

In [ ]:
def fill_values_soft_impute(M):
    mask = ~np.isnan(M)
    # Hold out 10% of observed entries for validation
    val_mask = mask & (np.random.rand(*M.shape) < 0.1)
    train_M = np.where(mask & ~val_mask, M, np.nan)

    best_loss, best_lam = np.inf, None
    for lam in [10**i for i in range(-1, 12)]:
        filled = fancyimpute.SoftImpute(shrinkage_value=lam).fit_transform(train_M)
        loss = np.mean((filled[val_mask] - M[val_mask])**2)
        if loss < best_loss:
            best_loss, best_lam = loss, lam

    M_filled = fancyimpute.SoftImpute(shrinkage_value=best_lam).fit_transform(M)
    print(best_lam)
    return M_filled

In [ ]:
def iterative_completion(M, n_iter=20):
    """Fill NaNs, do truncated SVD, re-fill, repeat."""
    mask = ~np.isnan(M)
    X = M.copy()
    col_means = np.nanmean(X, axis=0)
    X[~mask] = np.take(col_means, np.where(~mask)[1])

    for _ in range(n_iter):
        U, s, Vt = np.linalg.svd(X, full_matrices=False)
        rank, M_approx = fill_values_gavish_donoho(X)  # or use a fixed rank
        X[~mask] = M_approx[~mask]  # only update missing entries

    return X

def fill_values_gavish_donoho(M):
    # Estimate noise from median singular value
    n = M.shape[0]
    U, s, Vt = np.linalg.svd(M, full_matrices=False)
    noise_sigma = np.median(s) / (2.858 * np.sqrt(n))

    # Optimal threshold
    lam = (4 / np.sqrt(3)) * np.sqrt(n) * noise_sigma
    rank = np.sum(s > lam)
    return rank, (U[:, :rank] * s[:rank]) @ Vt[:rank]

In [ ]:
def fill_values_cross_validation(M):
    mask = ~np.isnan(M)
    # Hold out 15% of observed entries
    val_mask = mask & (np.random.rand(*M.shape) < 0.15)
    train_M = np.where(mask & ~val_mask, M, np.nan)

    # Initialize with column means
    def init_fill(X):
        X = X.copy()
        col_means = np.nanmean(X, axis=0)
        idx = np.where(np.isnan(X))
        X[idx] = np.take(col_means, idx[1])
        return X

    best_err, best_rank = np.inf, None
    for rank in range(1, 30):
        U, s, Vt = np.linalg.svd(init_fill(train_M), full_matrices=False)
        filled = (U[:, :rank] * s[:rank]) @ Vt[:rank]
        err = np.mean((filled[val_mask] - M[val_mask])**2)
        if err < best_err:
            best_err, best_rank = err, rank

    print(f"Best rank: {best_rank}")
    U, s, Vt = np.linalg.svd(init_fill(M), full_matrices=False)
    M_filled = (U[:, :best_rank] * s[:best_rank]) @ Vt[:best_rank]

In [ ]:
# fill_values_soft_impute(traffic_matrix)
rank, traffic_matrix_filled = fill_values_gavish_donoho(iterative_completion(traffic_matrix))
# fill_values_cross_validation(traffic_matrix)
print(rank)